# 01 — Treinamento YOLOv11n para Detecção de Componentes de Arquitetura

Pipeline completo conforme **Spec 002** do hackathon FIAP Fase 5.

| Etapa | Descrição |
|-------|-----------|
| 1 | Instalação de dependências |
| 2 | Configuração de seeds e caminhos |
| 3 | Validação do dataset |
| 4 | Treinamento YOLOv11n (fine-tuning) |
| 5 | Validação / métricas |
| 6 | Exportação ONNX |
| 7 | Inferência de exemplo |


## 1. Instalação de dependências

In [ ]:
# Instala as dependências exatas listadas em requirements-notebook.txt
import subprocess, sys

result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', '../requirements-notebook.txt', '-q'],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else 'OK')
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

## 2. Configuração — seeds, caminhos e hiperparâmetros (RF-05 / RNF-02)

In [ ]:
import random
import os
from pathlib import Path

import numpy as np
import torch

# ---------- Reproducibility (RNF-02) ----------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ---------- Paths ----------
REPO_ROOT   = Path('..').resolve()
DATASET_DIR = REPO_ROOT / 'dataset'
DATA_YAML   = DATASET_DIR / 'data.yaml'
MODELS_DIR  = REPO_ROOT / 'models'
MODELS_DIR.mkdir(exist_ok=True)

# ---------- Hyperparameters (RF-05) ----------
EPOCHS    = 100
IMGSZ     = 640
BATCH     = 16
PATIENCE  = 20          # early stopping
DEVICE    = 0 if torch.cuda.is_available() else 'cpu'
PROJECT   = str(REPO_ROOT / 'runs' / 'train')
RUN_NAME  = 'yolo11n_arch_components'

print(f'Device  : {DEVICE}')
print(f'CUDA    : {torch.cuda.is_available()}')
print(f'data.yaml: {DATA_YAML}')
print(f'Models  : {MODELS_DIR}')

## 3. Validação do dataset (CA-01 / RNF-01)

In [ ]:
import yaml
import collections

# --- Load data.yaml ---
with open(DATA_YAML) as f:
    cfg = yaml.safe_load(f)

class_names = cfg['names']
nc = len(class_names) if isinstance(class_names, dict) else len(class_names)
print(f'Classes ({nc}): {class_names}')

# --- Count images per split ---
splits = {'train': DATASET_DIR / 'train', 'val': DATASET_DIR / 'val', 'test': DATASET_DIR / 'test'}
for split, base in splits.items():
    imgs = list((base / 'images').glob('*.*'))
    lbls = list((base / 'labels').glob('*.txt'))
    print(f'{split:5s}: {len(imgs):4d} images | {len(lbls):4d} labels')

assert len(list((DATASET_DIR / 'train' / 'images').glob('*.*'))) >= 100, \
    'RF-01: dataset deve ter >=100 imagens de treino'

# --- Class distribution ---
class_counts: dict = collections.Counter()
for lbl_file in (DATASET_DIR / 'train' / 'labels').glob('*.txt'):
    for line in lbl_file.read_text().strip().splitlines():
        if line:
            class_counts[int(line.split()[0])] += 1

print('\nClass distribution (train labels):')
total_annotations = sum(class_counts.values())
for cls_id in sorted(class_counts):
    name = class_names[cls_id] if isinstance(class_names, dict) else class_names[cls_id]
    pct = class_counts[cls_id] / total_annotations * 100
    print(f'  [{cls_id:2d}] {name:<35s} {class_counts[cls_id]:6d}  ({pct:.1f}%)')
print(f'\nTotal annotations: {total_annotations}')

## 4. Treinamento — YOLOv11n fine-tuning (RF-05 / ADR-002)

In [ ]:
from ultralytics import YOLO

# Load YOLOv11n pretrained on COCO (backbone weights)
model = YOLO('yolo11n.pt')

print('Model summary (before fine-tuning):')
model.info()

# Fine-tune on the architecture-components dataset
results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    patience=PATIENCE,
    project=PROJECT,
    name=RUN_NAME,
    seed=SEED,
    # Augmentation — moderate, preserve diagram readability
    hsv_h=0.01,
    hsv_s=0.5,
    hsv_v=0.4,
    degrees=0.0,          # diagrams are rarely rotated
    translate=0.1,
    scale=0.4,
    fliplr=0.0,           # diagrams have left-right semantics
    mosaic=1.0,
    close_mosaic=10,
    # Logging
    verbose=True,
    plots=True,
    save=True,
    save_period=10,       # checkpoint every 10 epochs
    exist_ok=True,
)

print('\nTraining complete.')
print(f'Best model: {results.save_dir}/weights/best.pt')

## 5. Validação e métricas (RF-06 / RNF-03 / CA-02)

In [ ]:
import shutil

# Resolve path to best weights
best_pt = Path(results.save_dir) / 'weights' / 'best.pt'
assert best_pt.exists(), f'best.pt not found at {best_pt}'

# Load best model for evaluation
best_model = YOLO(str(best_pt))

# Validate on val split
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=IMGSZ,
    device=DEVICE,
    verbose=True,
)

map50      = val_metrics.box.map50
map50_95   = val_metrics.box.map
precision  = val_metrics.box.mp
recall     = val_metrics.box.mr

print('\n===== Validation metrics (val split) =====')
print(f'  mAP@0.5       : {map50:.4f}   (target > 0.75)')
print(f'  mAP@0.5:0.95  : {map50_95:.4f}   (target > 0.50)')
print(f'  Precision     : {precision:.4f}')
print(f'  Recall        : {recall:.4f}')

# Soft assertion — warn rather than crash so the rest of the pipeline runs
if map50 < 0.75:
    print(f'\n⚠️  WARNING: mAP@0.5 {map50:.4f} < 0.75 (RNF-03). Consider more epochs or data.')
else:
    print(f'\n✅  mAP@0.5 target met.')

# Validate on test split (CA-03)
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split='test',
    imgsz=IMGSZ,
    device=DEVICE,
    verbose=False,
)
print('\n===== Test split metrics =====')
print(f'  mAP@0.5       : {test_metrics.box.map50:.4f}')
print(f'  mAP@0.5:0.95  : {test_metrics.box.map:.4f}')

## 6. Exportação dos modelos (RF-06)

In [ ]:
# ---------- Copy best.pt to models/ ----------
dest_pt = MODELS_DIR / 'best.pt'
shutil.copy2(best_pt, dest_pt)
print(f'Saved PyTorch model : {dest_pt}')

# ---------- Export to ONNX ----------
best_model.export(
    format='onnx',
    imgsz=IMGSZ,
    opset=17,          # ONNX opset compatible with onnxruntime >=1.17
    simplify=True,
    dynamic=False,
)

# best.onnx lands next to best.pt in the run weights dir; copy it too
src_onnx  = best_pt.with_suffix('.onnx')
dest_onnx = MODELS_DIR / 'best.onnx'
if src_onnx.exists():
    shutil.copy2(src_onnx, dest_onnx)
    print(f'Saved ONNX model    : {dest_onnx}')
else:
    print(f'WARNING: ONNX file not found at {src_onnx}')

# ---------- Summary ----------
print('\n===== Exported artefacts =====')
for f in MODELS_DIR.iterdir():
    size_mb = f.stat().st_size / 1_048_576
    print(f'  {f.name:<20s}  {size_mb:.1f} MB')

## 7. Inferência de exemplo (CA-03)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2

# Pick a sample image from the test split
test_images = sorted((DATASET_DIR / 'test' / 'images').glob('*.*'))
assert test_images, 'No test images found'
sample_img = str(test_images[0])
print(f'Running inference on: {sample_img}')

# Run prediction
preds = best_model.predict(
    source=sample_img,
    conf=0.25,
    iou=0.45,
    imgsz=IMGSZ,
    verbose=False,
)

result = preds[0]
print(f'Detected {len(result.boxes)} objects')

# Visualise
img_bgr = cv2.imread(sample_img)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(img_rgb)
axes[0].set_title('Original')
axes[0].axis('off')

# Annotated
annotated = result.plot()
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title('Predicted detections')
axes[1].axis('off')

plt.suptitle(Path(sample_img).name, fontsize=12)
plt.tight_layout()
plt.show()

# Print detection summary
if isinstance(class_names, dict):
    id2name = class_names
else:
    id2name = {i: n for i, n in enumerate(class_names)}

print('\nDetections:')
for box in result.boxes:
    cls_id = int(box.cls.item())
    conf   = float(box.conf.item())
    print(f'  {id2name[cls_id]:<35s}  conf={conf:.3f}')

## 8. Relatório final

Preencha esta célula após o treinamento com os valores reais obtidos.


In [ ]:
print('===== FINAL REPORT =====')
print(f'Model       : YOLOv11n')
print(f'Dataset     : {DATA_YAML}')
print(f'Train imgs  : {len(list((DATASET_DIR/"train"/"images").glob("*.*")))}')
print(f'Val   imgs  : {len(list((DATASET_DIR/"val"/"images").glob("*.*")))}')
print(f'Test  imgs  : {len(list((DATASET_DIR/"test"/"images").glob("*.*")))}')
print(f'Classes     : {nc}')
print(f'Epochs      : {EPOCHS}  (patience={PATIENCE})')
print(f'imgsz       : {IMGSZ}')
print(f'mAP@0.5     : {map50:.4f}')
print(f'mAP@0.5:0.95: {map50_95:.4f}')
print(f'Precision   : {precision:.4f}')
print(f'Recall      : {recall:.4f}')
print(f'best.pt     : {dest_pt}')
print(f'best.onnx   : {dest_onnx}')